# Signpost-D teacher labels for shift_dev_v2: do labels from a ~4B model look more like the benchmark's gold?

The benchmark's gold is the average of three samples from a ~4B-class teacher model (its card: the teacher agrees with itself 73.5% of the time). Our training and referee labels come from strong Claude labellers. This notebook tests whether that difference matters **without training and without touching the benchmark**:

1. An open ~4B model (`TEACHER`; `TEACHER_MODE = "quick"`: thinking off, sampled at several temperatures, the copy from the one whose self-agreement is closest to the benchmark teacher's 0.735, chosen before any ranking; `"thinking"`: the 29 Oct run) answers every case of shift_dev_v2 `SAMPLES` times at moderate temperature, with all of a case's questions together. Each question's teacher gold is the average of its samples' one-hot answers, as the benchmark's gold is built.
2. The result is written as a **copy**, `checkpoints/teacher_labels/shift_dev_v2_teacher.jsonl`: the same cases and questions, with the gold replaced and the reviewers' gold kept in `meta.gold_reviewers`. The delivered file is never changed. On Colab the copy is pushed to `CKPT_REPO`.
3. Notebook 14 with `LABELS = "teacher"` then scores the 13 saved runs on the copy. **If the copy ranks them more like the benchmark than the reviewers' labels did (rank correlation 0.53), teacher-style labels are closer to the benchmark's gold**, and relabelled training is worth a run.

Reported here: how often the samples parse and finish, the teacher's **self-agreement** (against the benchmark teacher's 0.735), and how often its top answer agrees with the reviewers' gold, overall, by kind and by how sure the reviewers' gold is. Runs on Colab (A100) with vLLM; the teacher is only a labeller here, so it runs in bf16.

In [2]:
# Parameters

TEACHER = "Qwen/Qwen3.5-4B"            # an open ~4B model
TEACHER_MODE = "quick"                  # "quick": thinking off, plain sampling at each of QUICK_TEMPERATURES, the copy from the
                                        # temperature whose self-agreement is closest to TARGET_AGREEMENT | "thinking": thinking on (run of 29 Oct)
SAMPLES = 3                             # answers per case, averaged into the gold, as the benchmark's
TARGET_AGREEMENT = 0.735                # the benchmark teacher's self-agreement (its card); fixed before any ranking is seen
QUICK_TEMPERATURES = [0.7, 1.0, 1.3]    # quick mode: full-distribution sampling (no top-p / top-k)
QUICK_MAX_TOKENS = 256                  # quick mode: the answer lines only
TEMPERATURE, TOP_P, TOP_K = 0.6, 0.95, 20   # thinking mode
MAX_TOKENS = 8192                       # thinking mode: thinking plus answers, per case
SOURCE = "independent_v3/shift_dev_v2.jsonl"                      # under data/signpost_d; never changed
OUT = {"quick": "checkpoints/teacher_labels/shift_dev_v2_teacher_quick.jsonl",   # notebook 14, LABELS = "teacher_quick"
       "thinking": "checkpoints/teacher_labels/shift_dev_v2_teacher.jsonl"}[TEACHER_MODE]   # LABELS = "teacher"
CKPT_REPO = "skamalj/signpost-d-checkpoints"                       # private; on Colab the copy is pushed here

In [3]:
# Cell 1 — Setup: vLLM, the token, the data

!pip install -q vllm

import collections, getpass, importlib.util, json, os, random, re, statistics, subprocess
from pathlib import Path

ON_COLAB = importlib.util.find_spec("google.colab") is not None
HF_TOKEN, HF_TOKEN_FROM = os.environ.get("HF_TOKEN") or None, "the environment"
if HF_TOKEN is None and ON_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or None
        HF_TOKEN_FROM = "Colab secret 'HF_TOKEN'"
    except Exception:
        pass
if HF_TOKEN is None:                                  # the editor shows this prompt at the top of its window
    HF_TOKEN = getpass.getpass("Hugging Face token (write access, for the checkpoint repo): ").strip() or None
    HF_TOKEN_FROM = "the prompt"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Hugging Face token: {HF_TOKEN_FROM if HF_TOKEN else 'NOT FOUND'}")   # never the token itself

DATA_BRANCH = "signpost-typed-decisions"
DATA_DIR = Path("data")
if not (DATA_DIR / "signpost_d").exists():
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data"
cases = [json.loads(l) for l in open(DATA_DIR / "signpost_d" / SOURCE, encoding="utf-8")]
print(f"{len(cases)} cases, {sum(len(c['questions']) for c in cases)} questions from {SOURCE}")

Hugging Face token: the prompt
440 cases, 1938 questions from independent_v3/shift_dev_v2.jsonl


In [4]:
# Cell 2 — The prompt (the same question rendering as notebook 11, letters for options) and the answer parser

LETTERS = [chr(65 + i) for i in range(26)]
SYSTEM = ("You read a case and answer every decision question about it. Think it through, then end with a line "
          "'Answers:' followed by one line per question: its number, a colon and the letter of the option you choose, "
          "for example '1: B'.")
SYSTEM_QUICK = ("You read a case and answer every decision question about it. Reply with one line per question: its number, "
                "a colon and the letter of the option you choose, for example '1: B'. Nothing else.")
ANSWER_LINE = re.compile(r"^\s*\**\[?(\d+)\]?\**\s*[:.)]\s*\**\s*\(?([A-Z])\b", re.M)


def render(q):
    """As notebook 11: (option names in order, the gold's keys in the same order)."""
    cr = q.get("criteria") or {}
    if q["type"] == "noul":
        return [f"{k}: {cr.get(k) or ''}".rstrip(": ") for k in ("true", "false")], ["true", "false"]
    if q["type"] == "choice":
        return [k.replace("_", " ") + (f": {v}" if v else "") for k, v in cr.items()], list(cr)
    return [str(c) for c in cr], [str(i) for i in range(len(cr))]


def question_block(q):
    names, _ = render(q)
    if q["type"] == "noul":
        head = f"Statement: {q['instructions']}\nIs the statement true or false?"
    elif q["type"] == "score":
        head = f"Question: {q['instructions']}\nLevels, lowest first:"
    else:
        head = f"Question: {q['instructions']}\nOptions:"
    lines = []
    for i, nm in enumerate(names):
        pos = ""
        if q["type"] == "score":
            n = len(names)
            pos = f"level {i + 1} of {n}" + (" (lowest)" if i == 0 else " (highest)" if i == n - 1 else "") + ": "
        lines.append(f"{LETTERS[i]}. {pos}{nm}")
    return head + "\n" + "\n".join(lines)


def user_text(c):
    state = c["state"] if isinstance(c["state"], str) else json.dumps(c["state"], ensure_ascii=False)
    return "Case:\n" + state + "\n\n" + "\n\n".join(f"[{i + 1}] " + question_block(q) for i, q in enumerate(c["questions"].values()))


def parse_answers(text, n):
    """Letters per question from the text after the thinking: the last 'Answers:' block; questions missing there take
    their last 'n: X' anywhere after the thinking."""
    text = text.rsplit("</think>", 1)[-1]
    block = text.rsplit("Answers:", 1)[-1] if "Answers:" in text else ""
    got = {int(m.group(1)): m.group(2) for m in ANSWER_LINE.finditer(block)}
    if len(got) < n:
        got = {**{int(m.group(1)): m.group(2) for m in ANSWER_LINE.finditer(text)}, **got}
    return [got.get(i + 1) for i in range(n)]

In [5]:
# Cell 3 — The teacher answers every case SAMPLES times (vLLM, batched): once in thinking mode, or at each quick temperature

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

tok = AutoTokenizer.from_pretrained(TEACHER, token=HF_TOKEN)
thinking = TEACHER_MODE == "thinking"
prompts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM if thinking else SYSTEM_QUICK},
                                    {"role": "user", "content": user_text(c)}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=thinking) for c in cases]
print(f"mode {TEACHER_MODE} | prompt tokens: median {statistics.median(len(tok(p)['input_ids']) for p in prompts[:50])} (first 50 cases)")
llm = LLM(model=TEACHER, dtype="bfloat16", max_model_len=16384, gpu_memory_utilization=0.85, seed=0,
          limit_mm_per_prompt={"image": 0, "video": 0})           # text only: the vision tower is not used
if thinking:
    settings = {TEMPERATURE: SamplingParams(n=SAMPLES, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, max_tokens=MAX_TOKENS, seed=0)}
else:
    settings = {t: SamplingParams(n=SAMPLES, temperature=t, max_tokens=QUICK_MAX_TOKENS, seed=0) for t in QUICK_TEMPERATURES}
RUNS = {}
for t, params in settings.items():
    outs = llm.generate(prompts, params)
    finished = sum(o.finish_reason == "stop" for out in outs for o in out.outputs)
    n = sum(len(out.outputs) for out in outs)
    print(f"temperature {t}: {n} answers, finished {finished / n:.1%}")
    RUNS[t] = outs

mode quick | prompt tokens: median 1382.0 (first 50 cases)
(MainProcess pid=3104) INFO 10-29 16:15:47 [api_utils.py:286] non-default args: {'dtype': 'bfloat16', 'max_model_len': 16384, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'model': 'Qwen/Qwen3.5-4B'}
(MainProcess pid=3104) INFO 10-29 16:15:48 [model.py:733] Resolved architecture: Qwen3_5ForConditionalGeneration
(MainProcess pid=3104) INFO 10-29 16:15:48 [model.py:2152] Using max model len 16384


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

(MainProcess pid=3104) INFO 10-29 16:15:50 [model.py:951] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(MainProcess pid=3104) INFO 10-29 16:15:50 [model.py:1008] Disabled mm_prefix attention mode because multimodal inputs are configuration-disabled. Attention backends without mm_prefix support may now be selected.
(MainProcess pid=3104) INFO 10-29 16:15:50 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=3104) INFO 10-29 16:15:50 [config.py:688] Mamba cache mode is set to 'align' for Qwen3_5ForConditionalGeneration by default when prefix caching is enabled


Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Processed prompts: 100%|██████████| 1320/1320 [01:14<00:00, 17.60it/s, est. speed input: 26273.18 toks/s, output: 311.82 toks/s] 

temperature 1.0: 1320 answers, finished 100.0%


Rendering prompts:   0%|          | 0/440 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1320/1320 [01:15<00:00, 17.49it/s, est. speed input: 26113.82 toks/s, output: 349.44 toks/s]

temperature 1.3: 1320 answers, finished 99.4%


In [6]:
# Cell 4 — The teacher gold (the samples' answers averaged), self-agreement per setting, the chosen setting's copy

def teacher_gold(outs):
    """-> (relabelled cases, per-question rows, questions no sample answered)."""
    rows, out_cases, unparsed = [], [], 0
    for c, out in zip(cases, outs):
        qids = list(c["questions"])
        answers = [parse_answers(o.text, len(qids)) for o in out.outputs]
        gold, meta_t = {}, {}
        for j, qid in enumerate(qids):
            q = c["questions"][qid]
            names, keys = render(q)
            picks = [a[j] for a in answers if a[j] and LETTERS.index(a[j]) < len(keys)]
            rev = c["gold"][qid]["probabilities"]
            if picks:
                votes = collections.Counter(keys[LETTERS.index(p)] for p in picks)
                probs = {k: round(votes[k] / len(picks), 4) for k in keys}
            else:                                      # no sample answered it: keep the reviewers' gold, flagged
                unparsed += 1
                probs = dict(rev)
            gold[qid] = {"type": q["type"], "probabilities": probs}
            meta_t[qid] = {"samples": len(picks), "answers": [keys[LETTERS.index(p)] for p in picks]}
            rows.append({"kind": c["meta"]["kinds"].get(qid, "?"), "type": q["type"], "n": len(picks),
                         "all_agree": len(picks) == SAMPLES and len(set(picks)) == 1,
                         "pairs": [a == b for i, a in enumerate(picks) for b in picks[i + 1:]],
                         "agree_reviewers": bool(picks) and max(probs, key=probs.get) == max(rev, key=rev.get),
                         "rev_top_p": max(rev.values()), "teacher_top_p": max(probs.values())})
        out_cases.append((c, gold, meta_t))
    return out_cases, rows, unparsed


mean = lambda xs: round(statistics.mean(xs), 3) if xs else None
GOLD = {t: teacher_gold(outs) for t, outs in RUNS.items()}
agreement = {t: mean([p for r in rows for p in r["pairs"]]) for t, (_, rows, _) in GOLD.items()}
for t, (_, rows, unparsed) in GOLD.items():
    print(f"temperature {t}: self-agreement pairwise {agreement[t]} | all {SAMPLES} agree {mean([r['all_agree'] for r in rows])} | "
          f"mean teacher top {mean([r['teacher_top_p'] for r in rows])} | teacher top = reviewers' top {mean([r['agree_reviewers'] for r in rows])} | "
          f"unparsed {unparsed}")
chosen = min(agreement, key=lambda t: abs(agreement[t] - TARGET_AGREEMENT))
print(f"\nchosen: temperature {chosen} (self-agreement {agreement[chosen]}, target {TARGET_AGREEMENT}), chosen before any ranking")
out_cases, rows, unparsed = GOLD[chosen]
for name, key in (("kind", lambda r: r["kind"]), ("type", lambda r: r["type"]),
                  ("reviewers' gold", lambda r: "clear (>=0.85)" if r["rev_top_p"] >= 0.85 else "leaning (0.6-0.85)" if r["rev_top_p"] >= 0.6 else "split (<0.6)")):
    groups = collections.defaultdict(list)
    for r in rows:
        groups[key(r)].append(r)
    print(f"  teacher top = reviewers' top, by {name}: " + " | ".join(f"{k} {mean([r['agree_reviewers'] for r in v])} (n {len(v)})" for k, v in sorted(groups.items())))

out = Path(OUT)
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w", encoding="utf-8") as fh:
    for c, gold, meta_t in out_cases:
        meta = dict(c["meta"], gold_reviewers=c["gold"], teacher={"model": TEACHER, "mode": TEACHER_MODE, "samples": SAMPLES,
                                                                    "temperature": chosen, "self_agreement": agreement[chosen],
                                                                    "per_question": meta_t})
        fh.write(json.dumps(dict(c, gold=gold, meta=meta), ensure_ascii=False) + "\n")
print(f"wrote {out} ({len(out_cases)} cases)")
if ON_COLAB and CKPT_REPO:
    from huggingface_hub import HfApi
    HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=str(out), path_in_repo=str(out.relative_to("checkpoints")),
                                      repo_id=CKPT_REPO, repo_type="model", commit_message=f"teacher labels ({TEACHER_MODE}) for shift_dev_v2")
    print(f"pushed to {CKPT_REPO}")
print(f"first answer of the first case:\n{RUNS[chosen][0].outputs[0].text[-400:]}")

temperature 0.7: self-agreement pairwise 0.707 | all 3 agree 0.582 | mean teacher top 0.846 | teacher top = reviewers' top 0.668 | unparsed 0
temperature 1.0: self-agreement pairwise 0.647 | all 3 agree 0.493 | mean teacher top 0.814 | teacher top = reviewers' top 0.65 | unparsed 0
temperature 1.3: self-agreement pairwise 0.597 | all 3 agree 0.405 | mean teacher top 0.786 | teacher top = reviewers' top 0.631 | unparsed 0

chosen: temperature 0.7 (self-agreement 0.707, target 0.735), chosen before any ranking
  teacher top = reviewers' top, by kind: compare 0.555 (n 806) | infer 0.713 (n 279) | judge 0.607 (n 456) | read 0.937 (n 397)
  teacher top = reviewers' top, by type: choice 0.674 (n 666) | noul 0.742 (n 799) | score 0.535 (n 473)
  teacher top = reviewers' top, by reviewers' gold: clear (>=0.85) 0.697 (n 1495) | leaning (0.6-0.85) 0.632 (n 323) | split (<0.6) 0.408 (n 120)
wrote checkpoints/teacher_labels/shift_dev_v2_teacher_quick.jsonl (440 cases)
pushed to skamalj/signpost-d-